In [ ]:
# Импорт необходимых библиотек

import warnings
warnings.filterwarnings('ignore')

import os
import sys
import csv
import math
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from tqdm.auto import tqdm

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data, paths, loop
from src import threshold as threshold_mod

In [ ]:
# Конфигурация для модели

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = Path(r'path')
TRAIN_CSV = DATA_DIR / 'train.csv'
TEST_CSV = DATA_DIR / 'test_stage1' / 'test.csv'
paths.DATA_DIR = DATA_DIR

IMG_SIZE = 576
IN_CHANNELS = 5
BATCH_SIZE = 8
EPOCHS = 10
LR = 4e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
NUM_WORKERS = 0
USE_FORENSICS = True
USE_WINDOWS = True
ENCODER = 'timm-efficientnet-b3'

MAX_TRAIN_ROWS = None 
VAL_RATIO = 0.1
TARGET_CLEAN_FRACTION = 0.30

OUT_DIR = Path('out_path')
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUT_DIR / 'best_model.pth'

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# Модель и адаптер

def build_model():
    return smp.Unet(
        encoder_name=ENCODER,
        encoder_weights='imagenet',
        in_channels=IN_CHANNELS,
        classes=1,
        activation=None,
        aux_params=dict(pooling='avg', dropout=0.2, classes=1),
    )

full_model = build_model().to(DEVICE)

In [ ]:
# Метрики и подсчет лосса

loss_fn = loop.CombinedLoss(alpha=0.7, beta=0.3)

In [ ]:
# Данные и их обработка

rows = data.read_train_csv(TRAIN_CSV, DATA_DIR, max_rows=MAX_TRAIN_ROWS, seed=42)
flags = data.compute_has_mask_flags(rows)

tr_rows, tr_flags, val_rows, val_flags = data.stratified_split(rows, flags, VAL_RATIO, 42)
tr_rows = tr_rows + data.build_clean_example_rows(tr_rows, tr_flags, TARGET_CLEAN_FRACTION, 42)

val_rows = val_rows + data.orgl_clean_rows(val_rows)

train_ds = data.SegCropDataset(tr_rows, crop_size=IMG_SIZE, train=True, use_forensics=USE_FORENSICS)
val_ds = data.SegDataset(val_rows, IMG_SIZE, train=False, use_forensics=USE_FORENSICS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
valid_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
optimizer = torch.optim.AdamW(full_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

writer = SummaryWriter(str(OUT_DIR / 'tb'))
best_score = -float('inf')

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}, lr={optimizer.param_groups[0]["lr"]:.3e}')
    train_logs = loop.train_one_epoch(full_model, train_loader, loss_fn, optimizer, DEVICE, epoch, writer)
    val_logs = loop.evaluate(full_model, valid_loader, loss_fn, DEVICE)
    scheduler.step()

    score = 0.5 * val_logs['dice'] + 0.5 * val_logs['clean']
    writer.add_scalar('Val/dice', val_logs['dice'], epoch)
    writer.add_scalar('Val/clean', val_logs['clean'], epoch)
    writer.add_scalar('Val/loss', val_logs['loss'], epoch)
    writer.add_scalar('Metrics/Val_score', score, epoch)
    print(f"  train_loss={train_logs['loss']:.4f}  val_loss={val_logs['loss']:.4f}  "
          f"dice={val_logs['dice']:.4f}  clean={val_logs['clean']:.4f}  score={score:.4f}")

    if score > best_score:
        best_score = score
        torch.save({'model_state_dict': full_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'epoch': epoch + 1, 'best_score': best_score}, CKPT_PATH)

writer.close()

In [ ]:
# Подбор порога и Test Time Augmentation

@torch.no_grad()
def tta_probs(net, images):

    def seg_aux(x):
        seg, aux = net(x)
        
        return torch.sigmoid(seg), torch.sigmoid(aux)

    seg_acc, aux_acc = seg_aux(images)
    s, a = seg_aux(torch.flip(images, dims=[3]))
    seg_acc = seg_acc + torch.flip(s, dims=[3]); aux_acc = aux_acc + a
    s, a = seg_aux(torch.flip(images, dims=[2]))
    seg_acc = seg_acc + torch.flip(s, dims=[2]); aux_acc = aux_acc + a

    return (seg_acc / 3.0).cpu().numpy()[:, 0], (aux_acc / 3.0).cpu().numpy().reshape(-1)

def probs_to_native(prob, orig_h, orig_w):
    return cv2.resize(prob, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

def predict_windows(net, row, crop_size=576, overlap=0.33):
    img = data._load_rgb(row['chng'])
    h, w = img.shape[:2]
    step = int(crop_size * (1 - overlap))

    ys_list = list(range(0, max(1, h - crop_size + 1), step))
    xs_list = list(range(0, max(1, w - crop_size + 1), step))
    if ys_list[-1] + crop_size < h: ys_list.append(h - crop_size)
    if xs_list[-1] + crop_size < w: xs_list.append(w - crop_size)
    ys_list = [max(0, y) for y in ys_list]
    xs_list = [max(0, x) for x in xs_list]

    acc = np.zeros((h, w), dtype=np.float32)
    cnt = np.zeros((h, w), dtype=np.float32)
    aux_acc, aux_n = 0.0, 0

    for ys in ys_list:
        for xs in xs_list:
            y2, x2 = min(ys + crop_size, h), min(xs + crop_size, w)
            crop = img[ys:y2, xs:x2]
            ch, cw = crop.shape[:2]
            pad_h, pad_w = crop_size - ch, crop_size - cw
            if pad_h > 0 or pad_w > 0:
                crop = np.pad(crop, ((0, pad_h), (0, pad_w), (0, 0)))
            if USE_FORENSICS:
                ela = cv2.resize(data.forensics.compute_ela(crop), (crop_size, crop_size),
                                 interpolation=cv2.INTER_LINEAR)
                hp = cv2.resize(data.forensics.compute_highpass(crop), (crop_size, crop_size),
                                interpolation=cv2.INTER_LINEAR)
            else:
                ela = np.zeros((crop_size, crop_size), dtype=np.float32)
                hp = np.zeros((crop_size, crop_size), dtype=np.float32)
            stacked = data.stack_channels(crop, ela, hp)
            window = torch.from_numpy(stacked.transpose(2, 0, 1))[None].float().to(DEVICE)

            probs, auxs = tta_probs(net, window)
            prob = probs[0][:ch, :cw]
            acc[ys:y2, xs:x2] += prob
            cnt[ys:y2, xs:x2] += 1.0
            aux_acc += float(auxs[0]); aux_n += 1

    return acc / np.maximum(cnt, 1), aux_acc / max(aux_n, 1)

best = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
full_model.load_state_dict(best['model_state_dict'])
full_model.eval()

TUNE_MAX = 400
tune_rows = val_rows[:TUNE_MAX]
tune_loader = DataLoader(data.SegDataset(tune_rows, IMG_SIZE, train=False, use_forensics=USE_FORENSICS), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

prob_maps, aux_probs = [], []
if USE_WINDOWS:
    for row in tqdm(tune_rows, desc='window probs'):
        pmap, ap = predict_windows(full_model, row)
        prob_maps.append(pmap.astype(np.float16))
        aux_probs.append(ap)
else:
    with torch.no_grad():
        for batch in tqdm(tune_loader, desc='val probs'):
            images = batch['image'].to(DEVICE)
            probs, auxs = tta_probs(full_model, images)
            for i in range(probs.shape[0]):
                meta = {k: int(batch[k][i]) for k in ('orig_h', 'orig_w')}
                prob_maps.append(probs_to_native(probs[i], meta['orig_h'], meta['orig_w']).astype(np.float16))
                aux_probs.append(float(auxs[i]))

gt_maps = [(data.load_gt_mask_for_row(r) > 0.5).astype(np.uint8) for r in tune_rows]
chosen = threshold_mod.grid_search_threshold(prob_maps, gt_maps, aux_probs)

In [ ]:
# Инференс и сабмит модели

PRED_DIR = OUT_DIR / 'predictions'
PRED_DIR.mkdir(exist_ok=True)

test_rows = data.read_test_csv(TEST_CSV, DATA_DIR / 'test_stage1')
test_loader = DataLoader(data.TestDataset(test_rows, IMG_SIZE, use_forensics=USE_FORENSICS), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

sub_rows = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Predict Test'):
        images = batch['image'].to(DEVICE)
        probs, auxs = tta_probs(full_model, images)

        for i in range(probs.shape[0]):
            meta = {k: int(batch[k][i]) for k in ('orig_h', 'orig_w')}
            prob_native = probs_to_native(probs[i], meta['orig_h'], meta['orig_w'])
            binary = threshold_mod.postprocess_mask(prob_native, chosen['prob_threshold'], chosen['min_area'], chosen['reject_threshold'], float(auxs[i]), chosen.get('gate_threshold', 0.0))
            png = (binary * 255).astype(np.uint8)

            assert png.shape == (meta['orig_h'], meta['orig_w'])
            assert set(np.unique(png).tolist()).issubset({0, 255})

            rel = batch['chng_rel'][i]
            name = Path(rel).with_suffix('').as_posix().replace('/', '__') + '_pred.png'
            cv2.imwrite(str(PRED_DIR / name), png)
            sub_rows.append({'img_path': rel, 'prediction_path': f'{PRED_DIR.name}/{name}'})

with open(OUT_DIR / 'submission.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['img_path', 'prediction_path'])
    w.writeheader()
    w.writerows(sub_rows)

In [ ]:
# Диагностика 

y_true, y_pred = [], []
full_model.eval()

with torch.no_grad():
    for batch in tqdm(DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS), desc='Aux Diagnostics'):
        _, aux = full_model(batch['image'].to(DEVICE))
        y_pred.extend((torch.sigmoid(aux).cpu().numpy() > 0.5).astype(int).reshape(-1).tolist())
        y_true.extend(batch['has_manip'].numpy().astype(int).reshape(-1).tolist())

cm = [[0, 0], [0, 0]]
for t, p in zip(y_true, y_pred):
    cm[t][p] += 1

print('               pred Clean   pred Changed')
print(f'true Clean      {cm[0][0]:6d}        {cm[0][1]:6d}')
print(f'true Changed    {cm[1][0]:6d}        {cm[1][1]:6d}')
acc = (cm[0][0] + cm[1][1]) / max(1, len(y_true))
print(f'accuracy: {acc:.3f}  ({len(y_true)} строк)')